In [ ]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Paths

base_dir = (
   "path"
)


# E7 results

e7_results_dir = os.path.join(
    base_dir,
    "E7",
    "experiment",
    "DENOISING-TEST-Results-sigma25"
)

e7_denoised_dir = os.path.join(
    e7_results_dir,
    "denoised"
)

e7_noisy_dir = os.path.join(
    e7_results_dir,
    "noisy"
)


# E8 results

e8_results_dir = os.path.join(
    base_dir,
    "E8",
    "experiment",
    "DENOISING-TEST-Results-sigma25"
)

e8_denoised_dir = os.path.join(
    e8_results_dir,
    "denoised"
)

e8_noisy_dir = os.path.join(
    e8_results_dir,
    "noisy"
)


# Ground-truth test images

clean_dir = (
    "/home/sudad/project/"
    "image-70-20-10-2slices/test"
)


# Output directories

out_root = os.path.join(
    base_dir,
    "E7_vs_E8_ErrorMap_Analysis"
)

figures_dir = os.path.join(
    out_root,
    "figures"
)

os.makedirs(
    figures_dir,
    exist_ok=True
)






In [ ]:
# Verify image correspondence and noisy-image identity

def list_images(folder):
    extensions = [
        "*.png",
        "*.jpg",
        "*.jpeg",
        "*.tif",
        "*.tiff"
    ]

    files = []

    for ext in extensions:
        files.extend(
            glob.glob(
                os.path.join(folder, ext)
            )
        )

    return sorted(files)


# List images

clean_files = list_images(clean_dir)

e7_noisy_files = list_images(e7_noisy_dir)
e8_noisy_files = list_images(e8_noisy_dir)

e7_denoised_files = list_images(e7_denoised_dir)
e8_denoised_files = list_images(e8_denoised_dir)


print("Image Counts ")

print("Clean images       :", len(clean_files))
print("E7 noisy images    :", len(e7_noisy_files))
print("E8 noisy images    :", len(e8_noisy_files))
print("E7 denoised images :", len(e7_denoised_files))
print("E8 denoised images :", len(e8_denoised_files))



# Convert to dictionaries by filename

clean_dict = {
    os.path.basename(p): p
    for p in clean_files
}

e7_noisy_dict = {
    os.path.basename(p): p
    for p in e7_noisy_files
}

e8_noisy_dict = {
    os.path.basename(p): p
    for p in e8_noisy_files
}

e7_denoised_dict = {
    os.path.basename(p): p
    for p in e7_denoised_files
}

e8_denoised_dict = {
    os.path.basename(p): p
    for p in e8_denoised_files
}


# Compare filenames

clean_names = set(clean_dict.keys())
e7_noisy_names = set(e7_noisy_dict.keys())
e8_noisy_names = set(e8_noisy_dict.keys())
e7_denoised_names = set(e7_denoised_dict.keys())
e8_denoised_names = set(e8_denoised_dict.keys())


print("Filename Matching")

print(
    "Clean == E7 noisy      :",
    clean_names == e7_noisy_names
)

print(
    "Clean == E8 noisy      :",
    clean_names == e8_noisy_names
)

print(
    "Clean == E7 denoised   :",
    clean_names == e7_denoised_names
)

print(
    "Clean == E8 denoised   :",
    clean_names == e8_denoised_names
)

print(
    "E7 noisy == E8 noisy  :",
    e7_noisy_names == e8_noisy_names
)

print(
    "E7 denoised == E8 denoised:",
    e7_denoised_names == e8_denoised_names
)



# Find common filenames

common_names = sorted(
    clean_names
    & e7_noisy_names
    & e8_noisy_names
    & e7_denoised_names
    & e8_denoised_names
)


print(
    "Number of common images:",
    len(common_names)
)


if len(common_names) == 0:
    raise RuntimeError(
        "No common images were found among "
        "Clean, E7, and E8 folders."
    )


# Pixel-wise comparison of E7 and E8 noisy images

identical_count = 0
different_count = 0

max_difference_global = 0

different_images = []


for name in common_names:

    noisy_e7 = cv2.imread(
        e7_noisy_dict[name],
        cv2.IMREAD_GRAYSCALE
    )

    noisy_e8 = cv2.imread(
        e8_noisy_dict[name],
        cv2.IMREAD_GRAYSCALE
    )


    if noisy_e7 is None or noisy_e8 is None:
        raise RuntimeError(
            f"Could not read noisy image: {name}"
        )


    if noisy_e7.shape != noisy_e8.shape:

        different_count += 1

        different_images.append(
            {
                "name": name,
                "reason": "shape mismatch",
                "E7_shape": noisy_e7.shape,
                "E8_shape": noisy_e8.shape
            }
        )

        continue


    diff = np.abs(
        noisy_e7.astype(np.int16)
        - noisy_e8.astype(np.int16)
    )


    max_diff = int(
        diff.max()
    )


    max_difference_global = max(
        max_difference_global,
        max_diff
    )


    if max_diff == 0:

        identical_count += 1

    else:

        different_count += 1

        different_images.append(
            {
                "name": name,
                "reason": "pixel mismatch",
                "max_abs_difference": max_diff,
                "mean_abs_difference": float(diff.mean())
            }
        )




print("\n Noisy Image Identity Check ")

print(
    "Identical noisy images :",
    identical_count
)

print(
    "Different noisy images :",
    different_count
)

print(
    "Global max pixel difference:",
    max_difference_global
)

if different_count == 0:

    print(
        "\n All E7 and E8 noisy images are "
        "pixel-wise identical."
    )

    print(
        "Therefore, the E7 vs E8 comparison uses "
        "exactly the same noisy inputs."
    )

else:

    print(
        "\n Some noisy images are not identical."
    )

    print(
        "Do NOT proceed with the error-map comparison "
        "until this is investigated."
    )

    print(
        "\nFirst differences:"
    )

    for item in different_images[:10]:
        print(item)


In [ ]:
# Compute E7 vs E8 Error Statistics

results = []


for name in common_names:

    # Read images

    clean_u8 = cv2.imread(
        clean_dict[name],
        cv2.IMREAD_GRAYSCALE
    )

    e7_u8 = cv2.imread(
        e7_denoised_dict[name],
        cv2.IMREAD_GRAYSCALE
    )

    e8_u8 = cv2.imread(
        e8_denoised_dict[name],
        cv2.IMREAD_GRAYSCALE
    )


    if clean_u8 is None:
        raise RuntimeError(
            f"Could not read ground-truth image: {name}"
        )

    if e7_u8 is None:
        raise RuntimeError(
            f"Could not read E7 image: {name}"
        )

    if e8_u8 is None:
        raise RuntimeError(
            f"Could not read E8 image: {name}"
        )


    

    if not (
        clean_u8.shape
        == e7_u8.shape
        == e8_u8.shape
    ):
        raise RuntimeError(
            f"Shape mismatch for {name}: "
            f"GT={clean_u8.shape}, "
            f"E7={e7_u8.shape}, "
            f"E8={e8_u8.shape}"
        )


    # Convert uint8 images to [0,1]

    clean = clean_u8.astype(np.float32) / 255.0
    e7 = e7_u8.astype(np.float32) / 255.0
    e8 = e8_u8.astype(np.float32) / 255.0


    # Absolute error maps

    error_e7 = np.abs(e7 - clean)
    error_e8 = np.abs(e8 - clean)


    # MAE

    mae_e7 = float(
        np.mean(error_e7)
    )

    mae_e8 = float(
        np.mean(error_e8)
    )


    # RMSE

    rmse_e7 = float(
        np.sqrt(
            np.mean(
                (e7 - clean) ** 2
            )
        )
    )

    rmse_e8 = float(
        np.sqrt(
            np.mean(
                (e8 - clean) ** 2
            )
        )
    )


    # Store per-image statistics

    results.append(
        {
            "name": name,

            "MAE_E7": mae_e7,
            "MAE_E8": mae_e8,

            "Delta_MAE_E8_minus_E7": (
                mae_e8 - mae_e7
            ),

            "RMSE_E7": rmse_e7,
            "RMSE_E8": rmse_e8,

            "Delta_RMSE_E8_minus_E7": (
                rmse_e8 - rmse_e7
            )
        }
    )


# Create DataFrame

df_error = pd.DataFrame(
    results
)

if df_error.empty:
    raise RuntimeError(
        "No images were processed."
    )


# Save statistics

csv_path = os.path.join(
    out_root,
    "E7_vs_E8_error_statistics.csv"
)

df_error.to_csv(
    csv_path,
    index=False
)


# Global statistics


mean_mae_e7 = df_error["MAE_E7"].mean()
mean_mae_e8 = df_error["MAE_E8"].mean()

mean_rmse_e7 = df_error["RMSE_E7"].mean()
mean_rmse_e8 = df_error["RMSE_E8"].mean()

mean_delta_mae = df_error[
    "Delta_MAE_E8_minus_E7"
].mean()

mean_delta_rmse = df_error[
    "Delta_RMSE_E8_minus_E7"
].mean()


print(
    "\n Global Error Statistics "
)

print(
    f"Number of images: {len(df_error)}"
)

print()

print(
    f"E7 mean MAE  : {mean_mae_e7:.8f}"
)

print(
    f"E8 mean MAE  : {mean_mae_e8:.8f}"
)

print(
    f"Delta MAE    : {mean_delta_mae:+.8f}"
)

print()

print(
    f"E7 mean RMSE : {mean_rmse_e7:.8f}"
)

print(
    f"E8 mean RMSE : {mean_rmse_e8:.8f}"
)

print(
    f"Delta RMSE   : {mean_delta_rmse:+.8f}"
)

print()



print(
    csv_path
)



In [ ]:

# Common Global Visualization Scales


# Combine E7 and E8 absolute errors

combined_errors = np.concatenate(
    [
        global_error_e7,
        global_error_e8
    ]
)


# Common scale for |E7-GT| and |E8-GT|

# 99th percentile is used ONLY for visualization.
# It does not modify the numerical calculations.

error_vmax = float(
    np.percentile(
        combined_errors,
        99.0
    )
)



# Common symmetric scale for:
# Delta Error = |E8-GT| - |E7-GT|


diff_vmax = float(
    np.percentile(
        np.abs(global_diff),
        99.0
    )
)


if error_vmax <= 0:
    error_vmax = 1e-6

if diff_vmax <= 0:
    diff_vmax = 1e-6


# Print values



print(
    f"Absolute-error vmax [0,1] : "
    f"{error_vmax:.8f}"
)

print(
    f"Absolute-error vmax [0,255]: "
    f"{error_vmax * 255.0:.4f}"
)

print(
    f"Difference-map limit [0,1]: "
    f"±{diff_vmax:.8f}"
)

print(
    f"Difference-map limit [0,255]: "
    f"±{diff_vmax * 255.0:.4f}"
)



In [ ]:
# Select Q25, Q50, and Q75 representative cases
# based on Delta MAE = MAE_E8 - MAE_E7

delta_col = "Delta_MAE_E8_minus_E7"



# Compute quantiles


q25_value = float(
    df_error[delta_col].quantile(0.25)
)

q50_value = float(
    df_error[delta_col].quantile(0.50)
)

q75_value = float(
    df_error[delta_col].quantile(0.75)
)



# Select the image closest to each quantile


def closest_case_to_value(dataframe, column, target_value):

    idx = (
        dataframe[column] - target_value
    ).abs().idxmin()

    return dataframe.loc[idx]


case_q25 = closest_case_to_value(
    df_error,
    delta_col,
    q25_value
)

case_q50 = closest_case_to_value(
    df_error,
    delta_col,
    q50_value
)

case_q75 = closest_case_to_value(
    df_error,
    delta_col,
    q75_value
)


# Report selected cases

for label, target, case in [
    ("Q25", q25_value, case_q25),
    ("Q50", q50_value, case_q50),
    ("Q75", q75_value, case_q75)
]:
    print(
        f"{label}: "
        f"target={target:+.8f}, "
        f"image={case['name']}, "
        f"selected Delta MAE={case[delta_col]:+.8f}"
    )

In [ ]:
# Q50 Fixed ROI Statistics

# Fixed 80 x 80 ROI selected from the Q50 ground-truth image
x1, x2 = 75, 155
y1, y2 = 135, 215

q50_name = case_q50["name"]


# Load Q50 images

gt = read_gray01(
    clean_dict[q50_name]
)

e7 = read_gray01(
    e7_denoised_dict[q50_name]
)

e8 = read_gray01(
    e8_denoised_dict[q50_name]
)


# Extract the same ROI

gt_roi = gt[y1:y2, x1:x2]
e7_roi = e7[y1:y2, x1:x2]
e8_roi = e8[y1:y2, x1:x2]


# ROI MAE

mae_e7_roi = float(
    np.mean(
        np.abs(e7_roi - gt_roi)
    )
)

mae_e8_roi = float(
    np.mean(
        np.abs(e8_roi - gt_roi)
    )
)


# ROI RMSE

rmse_e7_roi = float(
    np.sqrt(
        np.mean(
            (e7_roi - gt_roi) ** 2
        )
    )
)

rmse_e8_roi = float(
    np.sqrt(
        np.mean(
            (e8_roi - gt_roi) ** 2
        )
    )
)


# Report

print("========== Q50 Fixed ROI Statistics ")

print(f"Image       : {q50_name}")
print(f"ROI         : x=[{x1},{x2}), y=[{y1},{y2})")
print(f"ROI size    : {x2-x1} x {y2-y1}")

print()

print(f"E7 ROI MAE  : {mae_e7_roi:.8f}")
print(f"E8 ROI MAE  : {mae_e8_roi:.8f}")
print(
    f"Delta MAE   : "
    f"{mae_e8_roi-mae_e7_roi:+.8f}"
)

print()

print(f"E7 ROI RMSE : {rmse_e7_roi:.8f}")
print(f"E8 ROI RMSE : {rmse_e8_roi:.8f}")
print(
    f"Delta RMSE  : "
    f"{rmse_e8_roi-rmse_e7_roi:+.8f}"
)



In [ ]:
# Q25 / Q50 / Q75 full-image error analysis
# with Q50 fixed ROI visualization

from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt
import numpy as np
import os


#  Fixed Q50 ROI


x1, x2 = 75, 155
y1, y2 = 135, 215

roi_width = x2 - x1
roi_height = y2 - y1



# Quantile cases


quantile_cases = [
    (
        "Q25",
        case_q25["name"],
        float(case_q25["Delta_MAE_E8_minus_E7"])
    ),
    (
        "Q50",
        case_q50["name"],
        float(case_q50["Delta_MAE_E8_minus_E7"])
    ),
    (
        "Q75",
        case_q75["name"],
        float(case_q75["Delta_MAE_E8_minus_E7"])
    )
]



# Helper

def load_case(name):

    gt = read_gray01(
        clean_dict[name]
    )

    noisy = read_gray01(
        e7_noisy_dict[name]
    )

    e7 = read_gray01(
        e7_denoised_dict[name]
    )

    e8 = read_gray01(
        e8_denoised_dict[name]
    )

    error_e7 = np.abs(
        e7 - gt
    )

    error_e8 = np.abs(
        e8 - gt
    )

    diff_error = (
        error_e8
        - error_e7
    )

    return (
        gt,
        noisy,
        e7,
        e8,
        error_e7,
        error_e8,
        diff_error
    )


# Figure layout

fig = plt.figure(
    figsize=(18, 12)
)

gs = fig.add_gridspec(
    nrows=4,
    ncols=9,

    width_ratios=[
        1, 1, 1, 1,
        1, 1,
        0.07,
        1,
        0.07
    ],

    hspace=0.12,
    wspace=0.08
)


axes = np.empty(
    (4, 7),
    dtype=object
)


# Main image axes
for row in range(4):

    for col in range(6):

        axes[row, col] = fig.add_subplot(
            gs[row, col]
        )

    axes[row, 6] = fig.add_subplot(
        gs[row, 7]
    )


# Shared colorbars
cax_error = fig.add_subplot(
    gs[:, 6]
)

cax_diff = fig.add_subplot(
    gs[:, 8]
)



# Column titles


titles = [
    "Ground Truth",
    "Noisy",
    "E7 (No GAN)",
    "E8 (GAN)",
    "|E7 − GT|",
    "|E8 − GT|",
    "Δ Error\n|E8−GT| − |E7−GT|"
]


for col, title in enumerate(titles):

    axes[0, col].set_title(
        title,
        fontsize=10.5,
        pad=5
    )


# Q25 / Q50 / Q75 full-image analysis

for row, (
    quantile_label,
    name,
    delta_mae
) in enumerate(quantile_cases):

    (
        gt,
        noisy,
        e7,
        e8,
        error_e7,
        error_e8,
        diff_error
    ) = load_case(name)


    images = [
        gt,
        noisy,
        e7,
        e8,
        error_e7,
        error_e8,
        diff_error
    ]


    for col in range(7):

        ax = axes[row, col]

        if col <= 3:

            ax.imshow(
                images[col],
                cmap="gray",
                vmin=0,
                vmax=1
            )

        elif col in [4, 5]:

            im_error = ax.imshow(
                images[col],
                cmap="inferno",
                vmin=0,
                vmax=error_vmax
            )

        else:

            im_diff = ax.imshow(
                images[col],
                cmap="coolwarm",
                vmin=-diff_vmax,
                vmax=diff_vmax
            )


        ax.set_xticks([])
        ax.set_yticks([])


    # Row labels
    axes[row, 0].set_ylabel(
        f"{quantile_label}\n"
        f"ΔMAE={delta_mae:+.6f}",
        fontsize=10
    )


    # --------------------------------------------------------
    # ROI rectangle ONLY on Q50 row
    # --------------------------------------------------------

    if quantile_label == "Q50":

        for col in [0, 1, 2, 3]:

            rect = Rectangle(
                (x1, y1),
                roi_width,
                roi_height,

                linewidth=1.5,
                edgecolor="red",
                facecolor="none"
            )

            axes[row, col].add_patch(
                rect
            )


# Q50 fixed ROI

q50_name = case_q50["name"]

(
    gt,
    noisy,
    e7,
    e8,
    error_e7,
    error_e8,
    diff_error
) = load_case(q50_name)


roi_images = [
    gt[y1:y2, x1:x2],
    noisy[y1:y2, x1:x2],
    e7[y1:y2, x1:x2],
    e8[y1:y2, x1:x2],
    error_e7[y1:y2, x1:x2],
    error_e8[y1:y2, x1:x2],
    diff_error[y1:y2, x1:x2]
]


for col in range(7):

    ax = axes[3, col]

    if col <= 3:

        ax.imshow(
            roi_images[col],
            cmap="gray",
            vmin=0,
            vmax=1,
            interpolation="nearest"
        )

    elif col in [4, 5]:

        ax.imshow(
            roi_images[col],
            cmap="inferno",
            vmin=0,
            vmax=error_vmax,
            interpolation="nearest"
        )

    else:

        ax.imshow(
            roi_images[col],
            cmap="coolwarm",
            vmin=-diff_vmax,
            vmax=diff_vmax,
            interpolation="nearest"
        )

    ax.set_xticks([])
    ax.set_yticks([])


axes[3, 0].set_ylabel(
    "Q50 fixed ROI",
    fontsize=10
)




# Shared colorbars


cbar_error = fig.colorbar(
    im_error,
    cax=cax_error
)

cbar_error.set_label(
    "Absolute error",
    fontsize=9
)

cbar_error.ax.tick_params(
    labelsize=8
)


cbar_diff = fig.colorbar(
    im_diff,
    cax=cax_diff
)

cbar_diff.set_label(
    "Δ absolute error",
    fontsize=9
)

cbar_diff.ax.tick_params(
    labelsize=8
)


# Save

final_png = os.path.join(
    figures_dir,
    "FINAL_E7_E8_ErrorMap_Analysis.png"
)

final_pdf = os.path.join(
    figures_dir,
    "FINAL_E7_E8_ErrorMap_Analysis.pdf"
)


plt.savefig(
    final_png,
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    final_pdf,
    bbox_inches="tight"
)


plt.show()


print("Saved PNG:")
print(final_png)

print()

print("Saved PDF:")
print(final_pdf)